# RG3 RH 불량 탐지
## 기존 1,182행만 사용하는 전처리와 모델링 노트북

위에서부터 실행합니다. **다른 파일을 읽거나 행을 추가하지 않습니다.**

- 원본 1,182행을 보존하고 RH 591행(정상 566, 불량 25)만 학습·평가합니다.
- LH 591행에는 점수나 정상 판정을 부여하지 않습니다. `AI 판정 대상 제외 · 기존 품질검사 필요`를 표시합니다.
- SMOTE, 행 복제, 언더샘플링을 하지 않습니다. 클래스 가중치와 검사 기준을 비교합니다.
- 개발: 10월 21~22일 RH 341행 / 불량 15건. 내부 보류 평가: 10월 23일 RH 250행 / 불량 10건.
- 기준 파일의 데이터 구조를 이미 살펴봤으므로 보류 구간을 완전히 보지 않은 외부 검증이라고 주장하지 않습니다.

**중요한 한계:** 입력 수치가 이미 표준화된 것으로 보이며 기존 스케일러의 학습 범위는 확인되지 않았습니다.
이 노트북의 전처리는 학습 폴드에서만 학습하지만, 기존 전처리의 누수를 소급해 제거하지는 못합니다.
압력·온도 수치를 실제 물리 단위로 해석하거나 서로 다른 표준화 센서의 차이를 실제 온도차로 해석하지 않습니다.


## 0 라이브러리 설치
처음 실행하는 환경에서만 아래 셀의 주석을 해제합니다. 설치 후 필요하면 커널을 재시작하세요.
`%pip`는 현재 주피터 커널에 설치합니다. 이후 저장되는 `environment_versions.json`으로 버전을 기록합니다.


In [ ]:
# 아래 줄 앞의 #을 지우고 실행하세요.
# %pip install pandas numpy scikit-learn matplotlib joblib


In [ ]:
from pathlib import Path
import json
import hashlib
import platform
import importlib.metadata
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    average_precision_score, precision_recall_curve,
    confusion_matrix, precision_score, recall_score,
    roc_auc_score, ConfusionMatrixDisplay,
)
import joblib

SEED = 42
DATA_PATH = Path(r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_rg3_다시되돌림.csv")
# 다른 컴퓨터에서는 위 경로만 바꾸세요. 아래 폴더는 주피터의 현재 작업 폴더에 생성됩니다.
OUTPUT_DIR = Path.cwd() / "rg3_rh_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TARGET_RECALL = 0.80  # 탐색용 목표. 보장된 성능이나 대회 공식 기준이 아닙니다.
TEST_START = pd.Timestamp("2020-10-23")
LH_NOTICE = "LH 불량 학습 사례가 없어 AI 판정 대상에서 제외됩니다. 기존 품질검사를 진행하세요."
print("데이터:", DATA_PATH)
print("결과 저장:", OUTPUT_DIR)


## 1 기준 파일 검증과 행 보존
`original_index`는 추적 ID이며 모델 입력이 아닙니다. 다른 데이터셋을 실수로 넣으면 중단하도록
행 수·라벨·날짜·제품명·방향별 건수를 검사합니다. 공정값이 같다는 이유로 LH/RH 행을 삭제하지 않습니다.


In [ ]:
SENSOR_COLUMNS = [
    "Injection_Time", "Filling_Time", "Plasticizing_Time", "Cycle_Time",
    "Clamp_Close_Time", "Cushion_Position", "Plasticizing_Position",
    "Clamp_Open_Position", "Max_Injection_Speed", "Max_Screw_RPM",
    "Average_Screw_RPM", "Max_Injection_Pressure", "Max_Switch_Over_Pressure",
    "Max_Back_Pressure", "Average_Back_Pressure",
    "Barrel_Temperature_1", "Barrel_Temperature_2", "Barrel_Temperature_3",
    "Barrel_Temperature_4", "Barrel_Temperature_5", "Barrel_Temperature_6",
    "Hopper_Temperature", "Mold_Temperature_3", "Mold_Temperature_4",
]

def load_and_validate(path):
    df = pd.read_csv(path)
    required = {"original_index", "PassOrFail", "TimeStamp", "PART_NAME", "RH_LH", *SENSOR_COLUMNS}
    assert required.issubset(df.columns), f"누락 컬럼: {required - set(df.columns)}"
    assert len(df) == 1182, "기준 파일은 1,182행이어야 합니다."
    assert df["original_index"].notna().all() and df["original_index"].is_unique
    assert df["PassOrFail"].notna().all()
    assert set(df["PassOrFail"].unique()) == {0, 1}
    assert df["PassOrFail"].value_counts().to_dict() == {0: 1157, 1: 25}
    original_ids = df["original_index"].tolist()
    original_labels = df.set_index("original_index")["PassOrFail"].copy()
    df["TimeStamp"] = pd.to_datetime(df["TimeStamp"], errors="raise")
    df["RH_LH"] = df["RH_LH"].astype("string").str.strip()
    df["PART_NAME"] = df["PART_NAME"].astype("string").str.strip()
    assert set(df["PART_NAME"]) == {"RG3 MOLD'G W/SHLD, LH", "RG3 MOLD'G W/SHLD, RH"}
    assert df["RH_LH"].isin(["RH", "LH"]).all()
    assert df["PART_NAME"].str[-2:].eq(df["RH_LH"]).all()
    assert df.groupby("RH_LH")["PassOrFail"].agg(["size", "sum"]).to_dict() == {
        "size": {"LH": 591, "RH": 591}, "sum": {"LH": 0, "RH": 25}
    }
    assert set(df["TimeStamp"].dt.strftime("%Y-%m-%d")) == {"2020-10-21", "2020-10-22", "2020-10-23"}
    assert not df.duplicated(["TimeStamp", "RH_LH"]).any()
    # 이 파일은 단일 설비 데이터이므로 시각을 생산 그룹키로 사용한다.
    df["cycle_group_id"] = df["TimeStamp"].dt.strftime("%Y-%m-%d %H:%M:%S")
    assert df.groupby("cycle_group_id").size().eq(2).all()
    assert df["cycle_group_id"].nunique() == 591
    for c in SENSOR_COLUMNS:
        df[c] = pd.to_numeric(df[c], errors="raise")
    # 무한대는 결측으로 처리하고 대치값은 학습 폴드에서만 계산한다.
    df[SENSOR_COLUMNS] = df[SENSOR_COLUMNS].replace([np.inf, -np.inf], np.nan)
    assert df.groupby("cycle_group_id")[SENSOR_COLUMNS].nunique(dropna=False).le(1).all().all()
    assert df["original_index"].tolist() == original_ids
    pd.testing.assert_series_equal(df.set_index("original_index")["PassOrFail"], original_labels)
    return df

data = load_and_validate(DATA_PATH)
SOURCE_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
display(data.groupby("RH_LH")["PassOrFail"].agg(total="size", defects="sum"))
display(pd.crosstab(data["TimeStamp"].dt.date, data["PassOrFail"]))
print("전체 행 보존:", len(data), "| 생산 그룹:", data["cycle_group_id"].nunique())


## 2 결측과 컬럼 선정
- ID, 라벨, 날짜, 제품명, LH/RH는 공정 모델의 입력에서 제외합니다.
- `Clamp_Open_Position`: 기존 파일에서 상수이므로 제외합니다.
- `Average_Screw_RPM`: 측정 정의 확인이 필요한 변수로 기준 모델에서 제외합니다. 현재 표준화 값만 보고 임의로 배율을 고치지 않습니다.
- 나머지 **22개 공정 변수**를 사용합니다. 자동 이상치 제거·클리핑·행 삭제는 하지 않습니다.
- `Reason` 및 추가 원본 컬럼은 이 노트북에서 가져오지 않습니다. 파일에 없는 값은 생성하지 않습니다.


In [ ]:
EXCLUDED_SENSORS = ["Clamp_Open_Position", "Average_Screw_RPM"]
FEATURES = [c for c in SENSOR_COLUMNS if c not in EXCLUDED_SENSORS]
assert len(FEATURES) == 22
assert not {"PassOrFail", "original_index", "RH_LH", "TimeStamp", "Reason"} & set(FEATURES)
audit = pd.DataFrame({
    "missing_count": data[SENSOR_COLUMNS].isna().sum(),
    "unique_count": data[SENSOR_COLUMNS].nunique(),
    "used_in_baseline": [c in FEATURES for c in SENSOR_COLUMNS],
})
display(audit)
data.to_csv(OUTPUT_DIR / "rg3_all_1182_preserved.csv", index=False, encoding="utf-8-sig")
audit.to_csv(OUTPUT_DIR / "column_audit.csv", encoding="utf-8-sig")


## 3 RH 개발 구간과 내부 보류 구간 분리
10월 23일은 모델과 임계값을 고른 뒤에만 평가합니다. 개발 구간의 그룹 교차검증은 무작위 분할이므로
시간적 일반화를 보장하지 않습니다. 별도의 과거→미래 점검도 수행합니다.


In [ ]:
rh = data.loc[data["RH_LH"].eq("RH")].sort_values(["TimeStamp", "original_index"]).reset_index(drop=True)
dev = rh.loc[rh["TimeStamp"] < TEST_START].reset_index(drop=True)
holdout = rh.loc[rh["TimeStamp"] >= TEST_START].reset_index(drop=True)
assert (len(rh), len(dev), len(holdout)) == (591, 341, 250)
assert (int(dev.PassOrFail.sum()), int(holdout.PassOrFail.sum())) == (15, 10)
assert set(dev.cycle_group_id).isdisjoint(set(holdout.cycle_group_id))
assert dev.TimeStamp.max() < holdout.TimeStamp.min()

X_dev, y_dev = dev[FEATURES], dev["PassOrFail"].astype(int)
X_test, y_test = holdout[FEATURES], holdout["PassOrFail"].astype(int)
splitter = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
folds = list(splitter.split(X_dev, y_dev, groups=dev["cycle_group_id"]))
for tr, va in folds:
    assert y_dev.iloc[tr].nunique() == y_dev.iloc[va].nunique() == 2
    assert set(dev.iloc[tr].cycle_group_id).isdisjoint(set(dev.iloc[va].cycle_group_id))
display(pd.DataFrame([
    {"fold": i, "train_n": len(tr), "train_defects": int(y_dev.iloc[tr].sum()),
     "valid_n": len(va), "valid_defects": int(y_dev.iloc[va].sum())}
    for i, (tr, va) in enumerate(folds, 1)
]))


## 4 전처리 파이프라인과 모델 후보
중앙값 대치와 상수 제거, 로지스틱 회귀의 표준화는 매 학습 폴드에서만 학습합니다.
로지스틱 회귀 6개와 깊이를 제한한 랜덤 포레스트 4개를 비교합니다.
가중치는 행 개수를 바꾸지 않습니다. `balanced`는 각 학습 폴드의 라벨 비율로 산정됩니다.


In [ ]:
def make_pipeline(classifier, scale=False):
    steps = [("impute", SimpleImputer(strategy="median")),
             ("constant", VarianceThreshold(threshold=0.0))]
    if scale:
        steps.append(("scale", StandardScaler()))
    steps.append(("model", classifier))
    return Pipeline(steps)

candidates = {}
weights = {"none": None, "w5": {0: 1, 1: 5}, "balanced": "balanced"}
for C in [0.1, 1.0]:
    for tag, weight in weights.items():
        candidates[f"LR_C{C}_{tag}"] = make_pipeline(
            LogisticRegression(C=C, class_weight=weight, solver="liblinear",
                               max_iter=3000, random_state=SEED), scale=True)
for depth in [2, 3]:
    for tag in ["none", "balanced"]:
        candidates[f"RF_d{depth}_{tag}"] = make_pipeline(
            RandomForestClassifier(n_estimators=200, max_depth=depth,
                min_samples_leaf=15, max_features="sqrt", class_weight=weights[tag],
                random_state=SEED, n_jobs=-1))
print("비교 후보:", len(candidates))


## 5 개발 구간에서 후보 비교
평균 AP가 높은 모델을 선택합니다. AP는 `average_precision_score`이며 사다리꼴 PR-AUC와 다릅니다.
여러 후보 중 선택된 교차검증 점수는 낙관적일 수 있으므로 최종 성능으로 제시하지 않습니다.
이 단계에서는 보류 구간의 정답이나 점수를 사용하지 않습니다.


In [ ]:
records, oof_by_model = [], {}
for name, template in candidates.items():
    oof = np.full(len(dev), np.nan)
    fold_ap = []
    for fold, (tr, va) in enumerate(folds, 1):
        fitted = clone(template).fit(X_dev.iloc[tr], y_dev.iloc[tr])
        scores = fitted.predict_proba(X_dev.iloc[va])[:, 1]
        oof[va] = scores
        ap = average_precision_score(y_dev.iloc[va], scores)
        fold_ap.append(ap)
        records.append({"model": name, "fold": fold, "AP": ap})
    assert np.isfinite(oof).all()
    oof_by_model[name] = oof

fold_results = pd.DataFrame(records)
comparison = fold_results.groupby("model")["AP"].agg(mean_AP="mean", std_AP="std")
comparison = comparison.reset_index().sort_values(["mean_AP", "model"], ascending=[False, True])
BEST_NAME = comparison.iloc[0]["model"]
best_template = candidates[BEST_NAME]
oof_scores = oof_by_model[BEST_NAME]
display(comparison)
print("개발 구간에서 선택한 모델:", BEST_NAME)
comparison.to_csv(OUTPUT_DIR / "development_model_comparison.csv", index=False, encoding="utf-8-sig")
fold_results.to_csv(OUTPUT_DIR / "development_fold_AP.csv", index=False, encoding="utf-8-sig")


## 6 개발용 임계값 선택
개발 OOF(각 행을 학습에서 제외하고 얻은 예측)에서 목표 재현율 이상인 임계값 중 검사량이 가장 작은 값을 고릅니다.
80%는 탐색용 기준입니다. 후보 선택에도 같은 개발 자료를 사용했으므로 독립 성능 추정이 아닙니다.
가중 모델의 점수는 보정된 불량 확률이 아닙니다. OOF 점수와 전체 개발 구간 재학습 점수의 척도가 달라질 수 있습니다.


In [ ]:
def classification_metrics(y, scores, threshold):
    y = np.asarray(y, dtype=int)
    scores = np.asarray(scores, dtype=float)
    pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"n": len(y), "defects": int(y.sum()), "threshold": float(threshold),
            "TP": int(tp), "FN": int(fn), "FP": int(fp), "TN": int(tn),
            "recall": recall_score(y, pred, zero_division=0),
            "precision": precision_score(y, pred, zero_division=0),
            "inspection_rate": float(pred.mean()),
            "AP": average_precision_score(y, scores) if y.sum() else np.nan}

threshold_table = pd.DataFrame([
    classification_metrics(y_dev, oof_scores, t) for t in np.unique(oof_scores)
])
eligible = threshold_table.loc[threshold_table.recall >= TARGET_RECALL]
choice = eligible.sort_values(["inspection_rate", "threshold"], ascending=[True, False]).iloc[0]
THRESHOLD = float(choice.threshold)
display(choice.to_frame("development_OOF_only"))
threshold_table.to_csv(OUTPUT_DIR / "development_thresholds.csv", index=False, encoding="utf-8-sig")


## 7 개발 구간의 시간 순서 점검
10월 22일 04시 이전 RH 144행(불량 5건)으로 학습하고 이후 RH 197행(불량 10건)을 평가합니다.
모델 선택과 겹치는 개발 자료이므로 참고 결과입니다. 결과가 나쁘더라도 보류 구간을 보고 조정하지 않습니다.


In [ ]:
early = dev["TimeStamp"] < pd.Timestamp("2020-10-22 04:00:00")
assert (int(early.sum()), int((~early).sum())) == (144, 197)
assert (int(y_dev[early].sum()), int(y_dev[~early].sum())) == (5, 10)
temporal_model = clone(best_template).fit(X_dev.loc[early], y_dev.loc[early])
temporal_scores = temporal_model.predict_proba(X_dev.loc[~early])[:, 1]
temporal_metrics = classification_metrics(y_dev.loc[~early], temporal_scores, THRESHOLD)
display(pd.DataFrame([temporal_metrics]))


## 8 최종 설정 고정 후 보류 평가
아래 셀을 실행한 뒤 10월23일의 결과를 보고 모델이나 임계값을 수정하면 이 구간도 개발에 사용한 것이 됩니다.
불량 10건 중 1건은 재현율 10%p이므로 비율과 건수를 함께 봅니다.


In [ ]:
final_model = clone(best_template).fit(X_dev, y_dev)
test_scores = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_scores >= THRESHOLD).astype(int)
test_metrics = classification_metrics(y_test, test_scores, THRESHOLD)
display(pd.DataFrame([test_metrics]))
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, test_pred, labels=[0, 1],
    display_labels=["Normal", "Defect"], ax=ax, colorbar=False)
ax.set_title("RH temporal holdout: Oct 23")
plt.tight_layout()
plt.show()
fig.savefig(OUTPUT_DIR / "holdout_confusion.png", dpi=150, bbox_inches="tight")


## 9 검사량별 불량 포착
상위 10%, 20%, 30%, 50%만 검사할 때 포착 건수를 확인합니다. 이는 고정 임계값 판정과 별개의 순위 평가입니다.
무작위 검사 기대 포착률은 K/N이며 실제 한 번의 무작위 실험 결과가 아닙니다.
같은 점수는 시간과 original_index 순으로 처리합니다. 실제 운영에서는 현재 검사 대기열 안에서만 순위를 매깁니다.


In [ ]:
def inspection_curve(frame, scores):
    ranked = frame[["original_index", "TimeStamp", "PassOrFail"]].copy()
    ranked["score"] = np.asarray(scores)
    ranked = ranked.sort_values(["score", "TimeStamp", "original_index"], ascending=[False, True, True])
    total_defects = int(ranked.PassOrFail.sum())
    rows = []
    for fraction in [0.10, 0.20, 0.30, 0.50, 1.00]:
        k = int(np.ceil(len(ranked) * fraction))
        hit = int(ranked.head(k).PassOrFail.sum())
        rows.append({"requested_fraction": fraction, "inspect_n": k, "actual_fraction": k/len(ranked),
                     "defects_caught": hit, "defects_missed": total_defects-hit,
                     "recall": hit/total_defects if total_defects else np.nan,
                     "precision": hit/k, "random_expected_recall": k/len(ranked),
                     "cutoff_tie_count": int(ranked.score.eq(ranked.iloc[k-1].score).sum())})
    return pd.DataFrame(rows)

curve = inspection_curve(holdout, test_scores)
display(curve)
curve.to_csv(OUTPUT_DIR / "holdout_inspection_curve.csv", index=False, encoding="utf-8-sig")


## 10 제품별 결과와 LH 안내
LH의 점수·예측 라벨은 결측으로 둡니다. 기존 실제 라벨 0은 원본 정답이지 AI 판정이 아닙니다.
RH 개발 구간은 OOF 결과, 보류 구간은 보류 예측을 저장합니다. 이 둘을 합쳐 독립 검증 성능으로 보고하지 않습니다.


In [ ]:
results = data[["original_index", "TimeStamp", "PART_NAME", "RH_LH", "PassOrFail"]].copy()
results = results.rename(columns={"PassOrFail": "actual_label"}).set_index("original_index")
results["risk_score"] = np.nan
results["predicted_label"] = pd.Series(pd.NA, index=results.index, dtype="Int64")
results["evaluation_scope"] = "LH_not_scored"
results["message"] = LH_NOTICE
for frame, scores, scope in [(dev, oof_scores, "development_OOF_selected_model"),
                              (holdout, test_scores, "temporal_holdout")]:
    ids = frame["original_index"].tolist()
    pred = (scores >= THRESHOLD).astype(int)
    results.loc[ids, "risk_score"] = scores
    results.loc[ids, "predicted_label"] = pred
    results.loc[ids, "evaluation_scope"] = scope
    results.loc[ids, "message"] = np.where(pred == 1, "RH 우선검사 권고", "RH 임계값 미만 · 정상 확정 아님")
results = results.reset_index()
assert len(results) == 1182
assert results.original_index.tolist() == data.original_index.tolist()
assert results.loc[results.RH_LH.eq("LH"), ["risk_score", "predicted_label"]].isna().all().all()
results.to_csv(OUTPUT_DIR / "all_1182_predictions_and_LH_notice.csv", index=False, encoding="utf-8-sig")
test_rows = results.loc[results.evaluation_scope.eq("temporal_holdout")].copy()
missed = test_rows.loc[(test_rows.actual_label == 1) & (test_rows.predicted_label == 0)]
false_alarm = test_rows.loc[(test_rows.actual_label == 0) & (test_rows.predicted_label == 1)]
display(missed)
missed.to_csv(OUTPUT_DIR / "holdout_missed_defects.csv", index=False, encoding="utf-8-sig")
false_alarm.to_csv(OUTPUT_DIR / "holdout_false_alarms.csv", index=False, encoding="utf-8-sig")


## 11 모델과 재현 정보 저장
저장 모델은 **RH 개발 구간 341행으로 학습한 보류 평가 모델**입니다. 전체 RH 재학습을 자동으로 하지 않습니다.
입력은 이 파일과 같은 표준화 체계를 전제로 합니다. 원단위 신규 센서값을 그대로 넣으면 안 됩니다.
`joblib` 파일은 직접 생성한 신뢰할 수 있는 파일만 불러오세요.


In [ ]:
bundle = {"model": final_model, "features": FEATURES, "threshold": THRESHOLD,
          "model_name": BEST_NAME, "trained_scope": "RH 2020-10-21 to 2020-10-22, n=341",
          "input_scale": "same scaling as supplied CSV; original scaler unavailable",
          "LH_notice": LH_NOTICE}
joblib.dump(bundle, OUTPUT_DIR / "rh_holdout_model.joblib")
versions = {name: importlib.metadata.version(name)
            for name in ["pandas", "numpy", "scikit-learn", "matplotlib", "joblib"]}
versions["python"] = platform.python_version()
manifest = {"source_file": DATA_PATH.name, "source_sha256": SOURCE_SHA256,
            "total_rows": len(data), "RH_rows": len(rh), "LH_rows": 591,
            "dev_ids": dev.original_index.tolist(), "holdout_ids": holdout.original_index.tolist(),
            "features": FEATURES, "excluded_sensors": EXCLUDED_SENSORS,
            "seed": SEED, "model": BEST_NAME, "threshold": THRESHOLD,
            "target_development_recall": TARGET_RECALL,
            "holdout_metrics": test_metrics}
for filename, value in [("run_manifest.json", manifest), ("environment_versions.json", versions)]:
    (OUTPUT_DIR / filename).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
split_rows = dev[["original_index", "cycle_group_id"]].copy()
split_rows["partition"] = "development"
split_rows["oof_fold"] = -1
for i, (_, va) in enumerate(folds, 1):
    split_rows.loc[va, "oof_fold"] = i
test_split = holdout[["original_index", "cycle_group_id"]].copy()
test_split["partition"] = "holdout"
test_split["oof_fold"] = pd.NA
pd.concat([split_rows, test_split], ignore_index=True).to_csv(
    OUTPUT_DIR / "RH_split_manifest.csv", index=False, encoding="utf-8-sig")
print("저장 완료:", OUTPUT_DIR)


## 12 결과 해석
1. **주요 결과:** 보류 구간 RH 불량 10건 중 몇 건을 잡았는지, 정상 몇 건을 검사 대상으로 올렸는지 보고합니다.
2. **검사 우선순위:** 같은 검사량에서 무작위 기대 수준보다 포착률이 높아지는지 확인합니다. 작은 표본의 차이를 확정 효과로 주장하지 않습니다.
3. **LH:** 점수와 라벨을 생성하지 않고 기존 검사를 안내합니다. LH를 정상으로 간주해 모델 정확도를 올리지 않습니다.
4. **한계:** 불량 25건, 짧은 관측 기간, 기존 표준화 범위 불명확, 실제 신규 생산 데이터 검증 부재가 있습니다.
5. **다음 개선:** 보류 점수를 반복 최적화하지 말고 실험 버전을 기록합니다. 행 추가·증강 없이 개발 구간에서만 가중치·변수 구성을 검토합니다.

노트북은 실행 계획과 코드입니다. 저장된 셀에 성능 결과가 없다면 아직 학습이 실행되지 않은 상태입니다.
